In [1]:
import os, json, torch
from typing import List, Dict, Any
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel


In [2]:
CAI_DIR = "sft_data_icai"
PREF_PROMPTS_PATH = os.path.join(CAI_DIR, "alert_pref_prompts.jsonl")  
BASE_MODEL = "mistralai/Mistral-7B-v0.1"
# Change to evaluate SFT or DPO aligned model
FT_MODEL_DIR = "models_icai/mistral-base_v01-sft"
#FT_MODEL_DIR = "models/final_policy"
NUM = 100                           
ANSWER_TOKENS = 512
TEMP = 0.5
if "policy" in FT_MODEL_DIR:
    OUT_DIR = "rlaif_data_icai"
    ALIGNED = "dpo"
else:
    OUT_DIR = CAI_DIR
    ALIGNED = "sft_mistral_base"
OUT_JSONL = f"{OUT_DIR}/compare_base_vs_{ALIGNED}_100_temp_{TEMP}.jsonl"

In [3]:
def load_jsonl(path: str) -> List[Dict[str, Any]]:
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

pref_items = load_jsonl(PREF_PROMPTS_PATH)
prompts = [it["prompt"] for it in pref_items if "prompt" in it and it["prompt"].strip()]
prompts = prompts[:NUM]
print(f"Using {len(prompts)} prompts from: {PREF_PROMPTS_PATH}")

Using 100 prompts from: sft_data_icai/alert_pref_prompts.jsonl


In [4]:
tok = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
if tok.pad_token_id is None:
    tok.pad_token = tok.eos_token

dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

tmp_base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=dtype, device_map="auto")
ft = PeftModel.from_pretrained(tmp_base, FT_MODEL_DIR)
ft.eval()

base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=dtype, device_map="auto")
base.eval()

def generate(model, prompt: str, max_new_tokens=ANSWER_TOKENS, temp=TEMP, top_p=1.0) -> str:
    if getattr(tok, "chat_template", None):
        input_ids = tok.apply_chat_template(
            [{"role": "user", "content": prompt}],
            add_generation_prompt=True,
            return_tensors="pt",
        ).to(model.device)
        attention_mask = torch.ones_like(input_ids)
    else:
        encoded = tok(prompt, return_tensors="pt")
        input_ids = encoded["input_ids"].to(model.device)
        attention_mask = encoded["attention_mask"].to(model.device)

    with torch.no_grad():
        if temp == "greedy":
            out = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                top_p=top_p,
                pad_token_id=tok.pad_token_id,
            )
        else:
            out = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=temp,
                top_p=top_p,
                pad_token_id=tok.pad_token_id,
            )
    gen = out[0][input_ids.shape[-1]:]
    return tok.decode(gen, skip_special_tokens=True).strip()

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [5]:
results = []
for i, p in enumerate(prompts, 1):
    base_out = generate(base, p)
    sl_out   = generate(ft, p)

    results.append({
        "idx": i,
        "prompt": p,
        "base_output": base_out,
        "tuned_output": sl_out,
    })
    if i % 10 == 0 or i == len(prompts):
        print(f"[{i}/{len(prompts)}]")

with open(OUT_JSONL, "w", encoding="utf-8") as f:
    for r in results:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print("Wrote side-by-side comparisons to:", OUT_JSONL)

[10/100]
[20/100]
[30/100]
[40/100]
[50/100]
[60/100]
[70/100]
[80/100]
[90/100]
[100/100]
Wrote side-by-side comparisons to: sft_data_icai/compare_base_vs_sft_mistral_base_100_temp_0.5.jsonl


In [6]:
prompt = "Explain what differential privacy is in one sentence."
inputs = tok(prompt, return_tensors="pt").to(base.device)

with torch.no_grad():
    logits_base = base(**inputs).logits
    logits_ft   = ft(**inputs).logits

max_diff = (logits_base - logits_ft).abs().max().item()
print("Max abs diff in logits:", max_diff)

Max abs diff in logits: 0.9375


In [7]:
print(type(ft))
print(ft.peft_config)          
print("Active adapters:", getattr(ft, "active_adapters", None))

<class 'peft.peft_model.PeftModelForCausalLM'>
{'default': LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.18.0', base_model_name_or_path='mistralai/Mistral-7B-v0.1', revision=None, inference_mode=True, r=16, target_modules={'k_proj', 'up_proj', 'q_proj', 'v_proj', 'down_proj', 'gate_proj', 'o_proj'}, exclude_modules=None, lora_alpha=32, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, use_dora=False, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, arrow_config=None, ensure_weight_tying=False)}
Active 

In [8]:
lora_params = [(n, p) for n, p in ft.named_parameters() if "lora" in n.lower()]
print("Number of LoRA params:", len(lora_params))

for name, param in lora_params[:20]:  
    print(name, "norm:", param.data.norm().item())

Number of LoRA params: 448
base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight norm: 2.3015170097351074
base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight norm: 0.05698057636618614
base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight norm: 2.311903476715088
base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight norm: 0.02870064228773117
base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight norm: 2.313950300216675
base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight norm: 0.023938078433275223
base_model.model.model.layers.0.self_attn.o_proj.lora_A.default.weight norm: 2.31064510345459
base_model.model.model.layers.0.self_attn.o_proj.lora_B.default.weight norm: 0.04679469019174576
base_model.model.model.layers.0.mlp.gate_proj.lora_A.default.weight norm: 2.314969301223755
base_model.model.model.layers.0.mlp.gate_proj.lora_B.default.weight norm: 0.08394143730401993
base_model.model.m

In [9]:
print("base id:", id(base))
print("ft   id:", id(ft))

base id: 137360326677216
ft   id: 137380335556640


In [10]:
for n, p in base.named_parameters():
    if "lora" in n.lower():
        print("BASE HAS LORA PARAM:", n, "norm:", p.data.norm().item())